### what is hugging face?
hugging face is like github for ai. just as github stores source code, hugging face stores ai resources.

- it provides:
1. pretrained ai models, public datasets, ai demo applications.
we can simply download the existing model and use it.

- hugging face has models, datasets, spaces.
- models: pretrained neural networks. ex: chatbots, translation, summarization, image generation, speech recognition.
- ex models: gpt, llama, mistral, whisper, bert.
--- 
datasets: they are collection of data used for training, fine-tuning, evaluation. ex: imdb movie reviews, wikipedia, common voice.
---
spaces: it allows anyone to host ai applications. a space can contain gradio app, streamlit app, static website.

- the hugging face platform is accessed through python libraries like transformers, datasets, huggingface_hub.

- transformers: provides pretrained models like text generation, chat, translation, summarization, classification, image generation, speech recognition.

In [3]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="gpt2"
)

print(generator("Artificial Intelligence is", max_new_tokens=30))

NameError: name 'torch' is not defined

In [ ]:
# datasets
from data_sets import load_dataset
dataset = load_dataset("imdb")
print(dataset)

In [ ]:
# hugging face hub
from huggingface_hub import list_models
models = list_models(limit=5)

for model in models:
    print(model.id)

- Google colab:
1. many ai models require gpus, most personal laptops have cpu. large ai models prefere gpus. google colab provides free cloud gpus.
- we use google servers.
- benefits: free gpu, free storage, jupyter notebook, no installation, runs in browser.

- https://colab.research.google.com
- create a new notebook.
- go to runtime, change the runtime type to gpu.

verify the gpu:
--- 

In [ ]:
import torch

print(torch.cuda.is_available())
print(torch.get_device_name(0))

### running stable diffusion and flux on google colab:
- these are img generation models.
- stable diffusion: generates images from text prompts.

In [ ]:
import torch
from diffusers import StableDiffusionPipeline

pipe = StableDiffusionPipeline.from_pretrained("runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16
)

pipe = pipe.to("cuda")
image = pipe(
    "A futuristic city at sunset, ultra realistic"
).images[0]

image.save("city.png")

- flux: newer text to image model, produces highly detailed and prompt faithful images.

"runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16
)


In [ ]:
import torch
from diffusers import DiffusionPipeline

pipe = DiffusionPipeline.from_pretrained(
    "black-forest-labs/FLUX.1-schnell",
    torch_dtype=torch.bfloat16
)

pipe.to("cuda")

image = pipe(
    "A majestic dragon flying over snowy mountains at sunrise"
).images[0]

image.save("dragon.png")

- after installation of transformers lib, the easiest way to use an ai model is throguh a pipeline.
- a pipeline is a high-level api, taht performs all complex steps automatically.

- instead of manually downloading the model, tokenizer, and writing inference code, we simply create a pipeline and start making predictions.


In [ ]:
from transformers import pipeline

generator=pipeline("text-generatoion",
                   model="gpt2")
result=generator("Artificial Intelligence is",
                 max_new_tokens=30)

print(result[0]["generated_text"])

- A pipeline acts like a ready-made ai application

### huggingface pipelines for sentiment analysis on colab t4 gpu:
- sentiment analysis identifies whether text expresses a positive, negative or neutral opinion.


In [ ]:
from transformers import pipeline

classifier = pipeline(
    "sentiment-analysis"
)
print("This course is amazing")


- one of the biggest strengths of hugging face is that changing the task often requires changing only the pipeline name.
- Named Entity Recognition: identifies important entities in a sentence.

ex: microsoft hired john in new york.
output:
microsoft -> organization
john -> person
new york -> location

In [ ]:
from transformers import pipeline

ner = pipeline(
    'ner', grouped_entities=True
)

text="Microsoft hired John in New York"
print(ner(text))


- question answering: answers questions using a provided context.

context: python is a programming language developed by guido van rossum.

question: who developed python?
output: guido van rossum.

In [ ]:
from transformers import pipeline
qa = pipeline("question-answering")

context = """ 
Python is a programming language developed by guido van rossum"""
question="who developed python?"

print((qa(question=question, context=context)))

In [ ]:

# for image classification
from transformers import pipeline
classifier = pipeline("image-classification")
result = classifier("dog.jpg")
print(result)

In [ ]:
from transformers import pipeline

transcriber = pipeline(
    "automatic-speech-recognition"
)

print(transcriber("speech.wav"))

from diffusers import DiffusionPipeline
import torch

pipe = DiffusionPipeline.from_pretrained(
    "black-forest-labs/FLUX.1-schnell",
    torch_dtype=torch.bfloat16
)

pipe.to("cuda")

image = pipe(
    "A futuristic city at sunset"
).images[0]

image.save("output.png")

tokenizers: how llms convert text to numbers?

- large language models cannot understand raw text.
- they only process numbers.
- tokenizer convert human readable text into numerical tokens before the model processes it.

ex:
- computers cannot directly understand words like: Artificial Intelligence
the tokenizer converts the sentence into smaller units called tokens.

each token is mapped to a unique integer:
Artificial  →  9542

Intelligence → 18022

is → 318

amazing → 4998

. → 13

the model processes these integers, not words

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")

text = "Artificial Intelligence is amazing."

tokens = tokenizer.tokenize(text)

print(tokens)

In [ ]:
# text to ids
ids = tokenizer.encode(text)

print(ids)
# ids to text
decoded = tokenizer.decode(ids)

print(decoded)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "meta-llama/Llama-3.1-8B-Instruct"
)
text = "Artificial Intelligence is amazing."

ids = tokenizer.encode(text)

print(ids)

decoded = tokenizer.decode(ids)

print(decoded)

### how chat templates work:
- chat models dont receive plain text. they receive structured conversations.

instead of just: Hello
<|begin_of_text|>

<|user|>

Hello

<|assistant|>
- these markers are called as special tokens, they help the model where the conversation begins, who is speaking, when the assistant should respond.


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "meta-llama/Llama-3.1-8B-Instruct"
)

messages = [
    {"role": "user", "content": "Explain AI"}
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print(prompt)

| Model      | Optimized For                         |
| ---------- | ------------------------------------- |
| Phi        | General reasoning with compact models |
| DeepSeek   | Reasoning and mathematics             |
| Qwen Coder | Programming and code generation       |


- everything we've seen so far ultimately feeds into a transformer neural network.

- text -> tokenizer -> token id (encoding) -> transformer neural network -> predicted token ids -> tokenizer -> generated output(decoding)

## what is a transformer?

- a transformer is a neural network architecture designed to understand relationships between tokens using self-attention.
- instead of reading text one word at a time, ot can consider all relevant tokens when generating the next token.

- input tokens -> embedding layer -> transformer blocks -> output layer -> next token

### what is quantization?
- llm models are usually stored using 16-bit or 32-bit numbers.

quantization reduces the precision of these numbers to save memory.

embedding layers: converts words or user ids into dense mathematical vectors of fixed size.

Sentence

↓

Tokenizer

↓

Token IDs

↓

Embedding Layer

↓

Transformer Blocks

↓

Output Layer

↓

Next Token

the model doesnt directly work with token ids.
each token id is mapped to a dense vector called an embedding.
ex: token id - 14711
[0.21, -0.46]....
these vectors capture semantic inforamtion, allowing the model to learn relationships between words.


### transformer blocks?
Input

↓

Self-Attention

↓

Feed-Forward Network

↓

Output

User Prompt
      ↓
Chat Template
      ↓
Tokenizer
      ↓
Token IDs
      ↓
Token Embeddings
      ↓
Transformer Layers (Self-Attention + Feed-Forward)
      ↓
Next Token Prediction
      ↓
Generated Token IDs
      ↓
Tokenizer Decode
      ↓
Final Response

- so farm we know that user prompt is given to tokenizer, tokenizer converts into token ids. each token id is converted into a dense vecor called an embedding.
- an embedding captures the semantic meaning of a token.
- tokens with similar meanings tend to have similar embedding vectors.

llama architecture: unlike encoder-decoder models such as t5, llama is a decoder only transformer. it predicts one token at a time.

In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("gpt2")

print(model.transformer.wte)

- once the embeddings are created, they pass through many decoder layers. each decoder model has 4 major components.
1. self attention: allows every word to look at other important words.

The cat sat on the mat because it was soft.

When predicting "it", the model asks:
Which previous words are important?

cat?

sat?

mat?

soft?
---
feed forward network: after attention, each token passes through a small neural network.
- this allows the model to transform information into richer representations.

- why non-linearity matters? without activation functions, stacking many layers would behave like a single linear transformation.
---
running open source llms: phi, gemma, qwen, and deepseek.

one advantage of hugging face is that the same code works with many different models.



In [ ]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model="microsoft/Phi-4-mini-instruct"
)

print(pipe("Explain Artificial Intelligence.", max_new_tokens=100)[0]["generated_text"])

In [ ]:
pipe = pipeline(
    "text-generation",
    model="google/gemma-3-4b-it"
)

In [ ]:
pipe = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-7B-Instruct"
)

### visualizing token by token inference in GPT models
- an llm does not generate an entire paragraph at once.
it predicts one token at a time.

prompt is: Artificial Intelligence
model predicts: is

the model will stop predicting until the condition is met.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2")

inputs = tokenizer("Artificial Intelligence", return_tensors="pt")

outputs = model.generate(
    **inputs,
    max_new_tokens=20
)

print(tokenizer.decode(outputs[0]))

## build a synthetic data generator using transformers pipelines